# 1. Data and Radius Graphs

In MACE (and other Message Passing Neural Networks), atomic structures are represented as spatial graphs. 
- **Nodes** are atoms, featuring atomic numbers ($Z$) and 3D positions ($R$).
- **Edges** connect atoms that are within a certain distance cutoff ($r_{max}$).

In this notebook, we use the `ase` (Atomic Simulation Environment) library to create an atomic structure and convert it into a PyTorch Geometric `Data` object.

In [1]:
import os
import sys
import torch
import ase
from ase.build import molecule

# Add the parent directory to the path so we can import src
sys.path.append(os.path.abspath('..'))

from src.data import atoms_to_pyg_data

### Creating a Sample Molecule

Let's build a simple Benzene (C6H6) molecule to visualize the graph structure.

In [2]:
# Load a sample molecule from ASE
atoms = molecule('C6H6')

print(f"Molecule: {atoms.get_chemical_formula()}")
print(f"Number of atoms: {len(atoms)}")

Molecule: C6H6
Number of atoms: 12


### Converting to PyTorch Geometric Data

We use our custom `atoms_to_pyg_data` function. It calculates the `edge_index` using a pure PyTorch radius graph implementation, meaning two atoms are connected if the distance between them is less than `cutoff`.

In [3]:
# Convert to PyG graph with a 4.0 Angstrom cutoff
cutoff = 4.0
graph = atoms_to_pyg_data(atoms, cutoff=cutoff)

print("PyTorch Geometric Data Object:")
print(graph)

print("\n--- Node Features ---")
print(f"Atomic Numbers (z) shape: {graph.z.shape}")
print(f"Positions (pos) shape: {graph.pos.shape}")

print("\n--- Edge Features ---")
print(f"Edge Connectivity (edge_index) shape: {graph.edge_index.shape}")
print(f"Edge Vectors (edge_vec) shape: {graph.edge_vec.shape}")
print(f"Edge Lengths (edge_len) shape: {graph.edge_len.shape}")

PyTorch Geometric Data Object:
Data(edge_index=[2, 114], pos=[12, 3], z=[12], edge_vec=[114, 3], edge_len=[114])

--- Node Features ---
Atomic Numbers (z) shape: torch.Size([12])
Positions (pos) shape: torch.Size([12, 3])

--- Edge Features ---
Edge Connectivity (edge_index) shape: torch.Size([2, 114])
Edge Vectors (edge_vec) shape: torch.Size([114, 3])
Edge Lengths (edge_len) shape: torch.Size([114])


Notice the `edge_index` shape is `[2, num_edges]`. For every edge, it stores the `[source, target]` atomic indices. 

The `edge_vec` is explicitly computed as $R_{target} - R_{source}$. These vectors will be passed into the Spherical Harmonics later to give our neural network an understanding of 3D geometry.